# MyTTS · подготовка книги к озвучке голосами персонажей (Google Colab)

Скрипт `mytts_book.py` из репозитория
[davnozdu/mytts-books](https://github.com/davnozdu/mytts-books) берёт EPUB, находит персонажей,
сопоставляет их через LLM и собирает файл `.mytts-book` для приложения MyTTS (5.0.4 Beta 2 или новее).

**Порядок работы:** запускайте ячейки сверху вниз (`Ctrl+F9` — все сразу).

Ключа нет нигде в файлах: ячейка 3 спросит его скрытым вводом при запуске, он живёт только в памяти
сессии Colab. Нужен только ключ LLM (DeepSeek или Ollama Cloud) и файл книги `.epub`.
**Если ключа нет — просто нажмите Enter:** книга разберётся и соберётся в файл `.mytts-book`
с разделами и отпечатками, но без персонажей, и MyTTS озвучит её стандартным голосом.

## 1. Настройки

`PROVIDER` — `deepseek` или `ollama`. Модель и адрес по умолчанию подставляются из настроек скрипта,
менять их не обязательно. `MIN_SPEAKER` / `MIN_MENTIONS` — порог, с которого персонаж получает свой
голос (иначе он попадает в общий голос «прочих»). `OUT_ON_DRIVE = True` складывает рабочую папку на
Google Диск — тогда после обрыва сессии Colab обработку можно продолжить, не теряя готовые ответы модели.

In [ ]:
PROVIDER     = "deepseek"   # "deepseek" или "ollama"
MODEL        = ""           # пусто = модель провайдера по умолчанию
ENDPOINT     = ""           # пусто = адрес провайдера по умолчанию (или свой сервер Ollama)
SCOPE        = "auto"       # auto (роман/сборник), book или section
THINK        = True         # размышление модели (для качественной сверки полезно)
PARALLEL     = 1            # одновременных запросов к LLM
MIN_SPEAKER  = 2            # свой голос: столько реплик в ремарках…
MIN_MENTIONS = 30           # …или столько упоминаний (главный герой без реплик)
OUTPUT       = ""           # пусто = имя файла .mytts-book по названию книги
OUT_ON_DRIVE = False        # True — рабочая папка на Google Диске (переживает обрыв сессии)

REPO_URL = "https://github.com/davnozdu/mytts-books.git"
REPO_DIR = "/content/mytts-books"

## 2. Установка (один раз за сессию)

Colab — обычный Linux с Python 3.11/3.12, поэтому установка та же, что в README проекта:
клон репозитория и зависимости (`pymorphy3`, `pymorphy3-dicts-ru`, `beautifulsoup4` — чистый Python,
без сборки). Обычно 10–20 секунд.

In [ ]:
import os, subprocess, sys, time

def run(cmd, **kw):
    "Показать команду и её вывод; упасть, если команда вернула ошибку."
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    return subprocess.run(cmd, check=True, **kw)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
else:
    print("репозиторий уже склонирован:", REPO_DIR, flush=True)

run([sys.executable, "-m", "pip", "install", "-q", "-U", "-r", os.path.join(REPO_DIR, "requirements.txt")])
print("\nПроверка установки:")
run([sys.executable, "-c", "import pymorphy3, bs4; print('pymorphy3', pymorphy3.__version__, '| bs4', bs4.__version__)"])
print("\nPython:", sys.version.split()[0])

## 3. Ключ LLM — спросит при запуске

Ключ нигде не хранится: ни в ноутбуке, ни в файлах, ни в Секретах Colab. Ячейка спрашивает его скрытым
вводом каждый раз при запуске, и он остаётся только в памяти этой сессии.

**Ключа нет?** Нажмите Enter на запросе — обработка пойдёт без LLM и соберёт книгу под стандартный голос.

In [ ]:
import os

ENV_NAME = "DEEPSEEK_API_KEY" if PROVIDER == "deepseek" else "OLLAMA_API_KEY"
HAVE_KEY = bool(os.environ.get(ENV_NAME))   # ключ уже задан в этой сессии (%env, os.environ и т. п.)

if HAVE_KEY:
    print("ключ уже задан в окружении сессии:", ENV_NAME)
else:
    print("Провайдер:", PROVIDER, "· нужен ключ", ENV_NAME)
    print("Пустая строка (Enter) — книга соберётся на стандартном голосе, без обращения к LLM.\n")
    from getpass import getpass
    value = (getpass(f"Ключ {ENV_NAME} (Enter — без ключа): ") or "").strip()
    if value:
        os.environ[ENV_NAME] = value
        HAVE_KEY = True

print("\nрежим:", "голоса персонажей (LLM)" if HAVE_KEY else "стандартный голос (без ключа и без LLM)")

## 4. Книга (EPUB)

Выберите любой из вариантов — подойдёт первый сработавший:

1. **Загрузить с компьютера** — `BOOK_URL = ""` и `BOOK_PATH = ""` (ячейка спросит файл).
2. **Уже лежит в Colab** — укажите путь, например `BOOK_PATH = "/content/drive/MyDrive/Идиот.epub"`.
3. **Ссылка** — `BOOK_URL = "https://…/book.epub"`.

In [ ]:
import os, urllib.request

BOOK_PATH = ""   # путь к .epub в Colab или на Google Диске ("" — выбрать иначе)
BOOK_URL  = ""   # прямая ссылка на .epub ("" — не использовать)

if BOOK_PATH:
    pass
elif BOOK_URL:
    BOOK_PATH = "/content/" + (os.path.basename(BOOK_URL.split("?")[0]) or "book.epub")
    urllib.request.urlretrieve(BOOK_URL, BOOK_PATH)
else:
    try:
        from google.colab import files
        BOOK_PATH = list(files.upload().keys())[0]
    except ImportError:
        raise SystemExit("Укажите BOOK_PATH или BOOK_URL (загрузка файлов доступна только в Colab).")

assert os.path.isfile(BOOK_PATH), f"нет файла: {BOOK_PATH}"
print("книга:", BOOK_PATH, f"({os.path.getsize(BOOK_PATH) / 1e6:.1f} МБ)")

## 5. Рабочая папка и запуск

Извлечение персонажей → запросы к LLM → проверка ответа → раздача голосов → сборка `.mytts-book`.
Вывод печатается по мере работы (по одному запросу на раздел). Без ключа этапы LLM пропускаются,
и файл собирается без персонажей — MyTTS озвучит книгу стандартным голосом.

Если сессия оборвалась — запустите ячейку снова: готовые ответы возьмутся из кэша, повторных
запросов к LLM не будет.

In [ ]:
import json, os, sys, time
from argparse import Namespace

def drive_mounted():
    for base in ("/content/drive/MyDrive", "/content/drive/My Drive"):
        if os.path.isdir(base):
            return base
    return None

base = drive_mounted() if OUT_ON_DRIVE else None
if OUT_ON_DRIVE and not base:
    from google.colab import drive
    drive.mount("/content/drive")
    base = drive_mounted()

WORK_DIR = os.path.join(base, "mytts-out") if base else "/content/out"
os.makedirs(WORK_DIR, exist_ok=True)
print("рабочая папка:", WORK_DIR, "(на Google Диске)" if base else "(живёт до перезапуска Colab)")

sys.path.insert(0, REPO_DIR)
import mytts_book as mb          # скрипт проекта: те же этапы, что и у команды process

VOICES_FILE = os.path.join(REPO_DIR, "voices", "silero_cis.json")
stage = lambda title: print(f"\n== {title}", flush=True)
started = time.time()

stage("1/4 Персонажи в тексте")
mb.extract(Namespace(book=BOOK_PATH, out=WORK_DIR, scope=SCOPE,
                     min_count=2, max_candidates=200, show=8, show_requests=5))

wanted = HAVE_KEY
if wanted:
    try:
        stage(f"2/4 Сопоставление в LLM ({PROVIDER})")
        mb.llm(Namespace(dir=WORK_DIR, provider=PROVIDER, model=MODEL or None, endpoint=ENDPOINT or None,
                         parallel=PARALLEL, redo=False, timeout=900, max_tokens=None, think=THINK))
        mb.apply(Namespace(dir=WORK_DIR, show=12, show_casts=3))
        stage("3/4 Раздача голосов персонажам")
        mb.voices(Namespace(dir=WORK_DIR, voices=VOICES_FILE, min_speaker=MIN_SPEAKER,
                            min_mentions=MIN_MENTIONS, max_shared=2, show=12, show_casts=3))
        VOICE_MODE = "llm"
    except Exception as e:
        print(f"\nLLM-этап не завершился: {type(e).__name__}: {str(e)[:300]}", flush=True)
        print("Собираю файл на стандартном голосе. Уже полученные ответы остались в кэше — "
              "запустите ячейку снова, они не потеряются.", flush=True)
        wanted = False

if not wanted:
    stage("2/4 Стандартный голос: этапы LLM пропущены")
    existing = os.path.join(WORK_DIR, "cast.json")
    ready = False
    if os.path.isfile(existing):
        try:
            ready = bool(json.load(open(existing, encoding="utf-8")).get("casts"))
        except Exception:
            ready = False
    if ready:
        print("в рабочей папке уже есть раздача голосов из прошлого запуска — беру её", flush=True)
        VOICE_MODE = "reused"
    else:
        data = mb.read_json(os.path.join(WORK_DIR, "candidates.json"))
        request = mb.read_json(os.path.join(WORK_DIR, "llm_request.json"))
        mb.save(WORK_DIR, "cast.json", {
            "book": data["book"], "scope": request["scope"], "input_sha256": mb.artifact_identity(WORK_DIR),
            "narrator": "author", "others": "other",
            "sections": [dict(s, cast=None) for s in data["sections"]], "casts": []})
        print("разделов без персонажей:", len(data["sections"]), flush=True)
        VOICE_MODE = "standard"

stage("4/4 Файл для MyTTS")
mb.export(Namespace(dir=WORK_DIR, output=OUTPUT or None, max_forms=24, keep_other=False))
print(f"\nГотово за {time.time() - started:.0f} с · режим: "
      + {"llm": "голоса персонажей", "reused": "голоса персонажей из прошлого запуска",
         "standard": "стандартный голос (без LLM)"}[VOICE_MODE], flush=True)
if VOICE_MODE == "standard":
    print("Персонажей в файле нет: MyTTS прочитает книгу стандартным голосом. Захотите голоса персонажей — "
          "запустите ячейку 3, введите ключ и запустите эту ячейку снова.", flush=True)

## 6. Результат

Итоговый файл `.mytts-book` в рабочей папке. Перенесите его на телефон и загрузите в MyTTS:
настройки LLM → мультиголос → «Книги с голосами персонажей» → «Загрузить файл книги».

С ключом внутри — персонажи с голосами, «прочие» (мужской и женский общие голоса) и повествование
голосом автора. Без ключа — разделы и отпечатки предложений без персонажей: книга для стандартного голоса.

In [ ]:
import glob, json, os

books = sorted(glob.glob(os.path.join(WORK_DIR, "**", "*.mytts-book"), recursive=True),
               key=os.path.getmtime, reverse=True)
if not books:
    print("файл .mytts-book не найден; проверьте вывод ячейки 5")
else:
    path = books[0]
    data = json.load(open(path, encoding="utf-8"))
    own = [c for cast in data.get("casts", []) for c in cast.get("characters", []) if c.get("voice_hint")]
    print("файл:", path, f"({os.path.getsize(path) / 1024:.1f} КБ)")
    print("разделов:", len(data.get("sections", [])), "| отпечатков:",
          sum(len(v) for v in data.get("fingerprints", {}).values()),
          "| с персональным голосом:", len(own))
    for c in own:
        print(f"    {c['voice_hint']:<16} {c['name']} ({c['gender']}, реплик {c['speaker']}, упоминаний {c['mentions']})")
    if not own:
        print("    персонажей нет — файл собран под стандартный голос книги")
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        print("\nСкачать в Colab: файл появится на Google Диске (OUT_ON_DRIVE) или скачайте вручную.")

## 7. Не обязательно: тонкая настройка голосов

Нужно только для запуска с ключом. Если персональных голосов в файле слишком много или слишком мало,
поправьте `MIN_SPEAKER` / `MIN_MENTIONS` в ячейке 1 и запустите эту ячейку — результат пересоберётся
**без новых запросов к LLM** (ответы уже в кэше).

- `--min-speaker N` — сколько реплик в ремарках нужно для собственного голоса;
- `--min-mentions N` — или сколько упоминаний (для главного героя без реплик);
- `--max-shared N` — предел общих абзацев для общего голоса «прочих»;
- `--keep-other` — оставить в файле и второстепенных персонажей («прочие»).

In [ ]:
import json, os, subprocess, sys

cast_file = os.path.join(WORK_DIR, "cast.json")
ready = os.path.isfile(cast_file) and json.load(open(cast_file, encoding="utf-8")).get("casts")

if not ready:
    print("Раздачи голосов нет — книга собрана на стандартном голосе, настраивать нечего.")
    print("Чтобы получить голоса персонажей, запустите ячейку 3 с ключом и затем ячейку 5.")
else:
    def step(command, *args):
        "Запустить этап mytts_book.py для рабочей папки."
        cmd = [sys.executable, os.path.join(REPO_DIR, "mytts_book.py"), command, WORK_DIR, *args]
        print("$", " ".join(cmd[1:]), flush=True)
        subprocess.run(cmd, cwd=REPO_DIR, check=True)

    step("voices", "--min-speaker", str(MIN_SPEAKER), "--min-mentions", str(MIN_MENTIONS), "--show-casts", "1")
    step("export")

## Заметки

- **Ключ.** Спрашивается при каждом запуске (ячейка 3) и живёт только в памяти сессии — ни в ноутбуке,
  ни на Диске, ни в Секретах Colab он не сохраняется. Обработка идёт через API выбранного провайдера
  (`api.deepseek.com` или `ollama.com`); Colab такие запросы разрешает.
- **Без ключа.** Книга всё равно разбирается: получается `.mytts-book` с разделами и отпечатками
  предложений и пустым списком персонажей — MyTTS читает её стандартным голосом. Позже можно ввести
  ключ и запустить ячейку 5 снова: кэш разбора сохранится, повторно платить не придётся.
- **Приватность.** В облако уходит только текст книги — так же, как если запускать скрипт дома.
- **Время.** Запросы идут по одному разделу, чтобы не перегружать предел частоты провайдера.
  Большой роман — обычно 10–40 минут; ноутбук можно закрыть, вкладку Colab — нет (сессия завершится).
  Бесплатный Colab отключает среду после ~90 минут простоя, `/content` при этом стирается:
  `OUT_ON_DRIVE = True` сохраняет рабочую папку и кэш ответов на Google Диске.